In [ ]:
import pandas as pd
import seaborn as sns
import plotly.express as px



df = pd.read_csv("sport_car_price_cleaned.csv")
df.head



In [ ]:
df.columns


In [ ]:
import plotly.express as px
import pandas as pd

df = pd.read_csv("sport_car_price_cleaned.csv")

df["engine_size_l"] = df["engine_size_l"].astype(str).str.extract(r'(\d+\.?\d*)')
df["engine_size_l"] = pd.to_numeric(df["engine_size_l"], errors='coerce')
df["price_usd"] = df["price_usd"].astype(str).str.replace(",", "")
df["price_usd"] = pd.to_numeric(df["price_usd"], errors='coerce')

df_clean = df.dropna(subset=["engine_size_l", "price_usd", "horsepower"]).copy()
df_clean["price_usd"] = df_clean["price_usd"].round(0)
df_clean["engine_size_l"] = df_clean["engine_size_l"].round(1)

low = df_clean["engine_size_l"].quantile(0.025)
high = df_clean["engine_size_l"].quantile(0.975)
df_filtered = df_clean[(df_clean["engine_size_l"] >= low) & (df_clean["engine_size_l"] <= high)]

y_min = df_filtered["engine_size_l"].min() * 0.8
y_max = df_filtered["engine_size_l"].max() * 1.07
x_min = df_filtered["horsepower"].min() * 0.85
x_max = df_filtered["horsepower"].max() * 1.08

fig1 = px.scatter(
    df_filtered,
    x="horsepower",
    y="engine_size_l",
    size="engine_size_l",
    color="price_usd",
    hover_data={
        "car_make": True,
        "car_model": True,
        "year": True,
        "horsepower": True,
        "engine_size_l": True,
        "price_usd": ":$,.0f"
    },
    title="Engine Size vs Horsepower — Colored by Price",
    labels={
        "horsepower": "Horsepower (HP)",
        "engine_size_l": "Engine Size (L)",
        "price_usd": "Price (USD)"
    },
    color_continuous_scale=px.colors.sequential.Plasma,
    opacity=0.75,
    size_max=45,
    template="simple_white"
)

fig1.update_layout(
    xaxis=dict(range=[x_min, x_max]),
    yaxis=dict(range=[y_min, y_max]),
    title_font_size=20,
    title_x=0.5,
    margin=dict(l=40, r=40, t=60, b=40),
    coloraxis_colorbar=dict(
        title="Price (USD)",
        tickprefix="$",
        ticks="outside"
    )
)

fig1.show()


# This plot shows how horsepower and engine size typically rise together, but not linearly with price.


Mid-range performance (300–400 HP, 3.0–4.0L) offers high value, avoiding premium price spikes.


Outliers suggest exotic brands or inefficient power-cost tradeoffs.

In [ ]:
import plotly.express as px

# Filter for affordable-ish high-end cars
filtered_df = df[df["price_usd"] <= 200000].copy()

# Round for readability
filtered_df["zero_to_60mph_s"] = filtered_df["zero_to_60mph_s"].round(2)

# Sort brands by median 0-60 time (fastest first)
order = (
    filtered_df.groupby("car_make")["zero_to_60mph_s"]
    .median()
    .sort_values()
    .index
)

# Create enhanced box plot
fig_box = px.box(
    filtered_df,
    x="car_make",
    y="zero_to_60mph_s",
    category_orders={"car_make": order},
    color="car_make",
    labels={
        "zero_to_60mph_s": "0–60 mph Time (seconds)",
        "car_make": "Car Brand"
    },
    title=" 0–60 mph Acceleration Time by Car Brand (Under $200K)",
    template="plotly_white"
)

# Update layout
fig_box.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(family="Arial", size=14),
    xaxis_title="",
    xaxis_tickangle=-45,
    showlegend=False,
    margin=dict(l=40, r=40, t=60, b=80)
)

fig_box.show()


In [ ]:
import plotly.express as px

# Optional: focus on affordable supercars
filtered_df = df[df["price_usd"] <= 300000].copy()

# Round for clarity
filtered_df["torque_lb_ft"] = filtered_df["torque_lb_ft"].round(0)
filtered_df["price_usd"] = filtered_df["price_usd"].round(0)
filtered_df["zero_to_60mph_s"] = filtered_df["zero_to_60mph_s"].round(2)

# Build 3D scatter plot
fig_3d = px.scatter_3d(
    filtered_df,
    x="torque_lb_ft",
    y="zero_to_60mph_s",
    z="price_usd",
    color="car_make",
    hover_data={
        "car_model": True,
        "year": True,
        "horsepower": True,
        "price_usd": ":$,.0f"
    },
    labels={
        "torque_lb_ft": "Torque (lb-ft)",
        "zero_to_60mph_s": "0–60 mph (s)",
        "price_usd": "Price (USD)"
    },
    title=" Acceleration vs Torque vs Price — 3D View by Car Brand",
    template="plotly_white"
)

# Tweak layout
fig_3d.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(family="Arial", size=14),
    margin=dict(l=10, r=10, t=60, b=10),
    scene_camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
    legend=dict(itemsizing='constant')
)

fig_3d.show()


In [ ]:
import plotly.express as px

# Filter for cars under $200k
filtered_df = df[df["price_usd"] <= 200000].copy()

# Round for clarity
filtered_df["horsepower"] = filtered_df["horsepower"].round(0)
filtered_df["price_usd"] = filtered_df["price_usd"].round(0)

# Create enhanced scatter plot
fig_scatter = px.scatter(
    filtered_df,
    x="horsepower",
    y="price_usd",
    color="car_make",
    hover_data={
        "car_model": True,
        "year": True,
        "horsepower": True,
        "price_usd": ":$,.0f"
    },
    labels={
        "horsepower": "Horsepower (HP)",
        "price_usd": "Price (USD)",
        "car_make": "Brand"
    },
    title=" Horsepower vs Price — Sub-$200K Performance Cars",
    template="plotly_white"
)

# Increase point size & tweak layout
fig_scatter.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=0.5, color='gray')))
fig_scatter.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(family="Arial", size=14),
    margin=dict(l=40, r=40, t=60, b=40),
    legend_title_text='Brand'
)

fig_scatter.show()


In [ ]:
import plotly.express as px

# Prepare data
filtered_df = df[df["price_usd"] <= 200000].copy()
filtered_df["horsepower"] = filtered_df["horsepower"].round(0)
filtered_df["price_usd"] = filtered_df["price_usd"].round(0)
filtered_df["performance_ratio"] = (filtered_df["horsepower"] / (filtered_df["price_usd"] / 1000)).round(2)

# Top 5 efficient cars
top5 = filtered_df.nlargest(5, "performance_ratio")

# Build scatter plot
fig = px.scatter(
    filtered_df,
    x="horsepower",
    y="price_usd",
    color="performance_ratio",
    color_continuous_scale=px.colors.sequential.Turbo,
    hover_data={
        "car_make": True,
        "car_model": True,
        "year": True,
        "horsepower": True,
        "price_usd": ":$,.0f",
        "performance_ratio": True
    },
    labels={
        "horsepower": "Horsepower (HP)",
        "price_usd": "Price (USD)",
        "performance_ratio": "HP per $1K"
    },
    title=" Horsepower vs Price — Colored by Performance Efficiency",
    template="simple_white"
)

# Style tweaks
fig.update_traces(
    marker=dict(size=10, opacity=0.75, line=dict(width=0.6, color='rgba(50,50,50,0.4)'))
)

fig.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(family="Segoe UI", size=13),
    margin=dict(l=60, r=60, t=60, b=60),
    xaxis=dict(
        gridcolor="rgba(220,220,220,0.3)",
        zeroline=False
    ),
    yaxis=dict(
        gridcolor="rgba(220,220,220,0.3)",
        tickprefix="$",
        separatethousands=True,
        zeroline=False
    ),
    coloraxis_colorbar=dict(
        title="HP per $1K",
        ticksuffix=" HP",
        tickformat=".2f"
    )
)

# Annotate top 5
for _, row in top5.iterrows():
    fig.add_annotation(
        x=row["horsepower"],
        y=row["price_usd"],
        text=row["car_model"],
        showarrow=True,
        arrowhead=1,
        ax=30,
        ay=-40,
        font=dict(size=10, color="black"),
        bgcolor="white",
        opacity=0.9
    )

fig.show()


In [ ]:
import plotly.express as px

# Step 1: Group, Round, and Filter
avg_price_make = df.groupby("car_make")["price_usd"].mean().reset_index()
avg_price_make["price_usd"] = avg_price_make["price_usd"].round(0)
avg_price_make = avg_price_make[avg_price_make["price_usd"] <= 200000]  # 🚫 remove ultra-luxury
avg_price_make = avg_price_make.sort_values("price_usd", ascending=False)

# Step 2: Plot
fig_bar = px.bar(
    avg_price_make,
    x="car_make",
    y="price_usd",
    text="price_usd",
    labels={
        "car_make": "Car Brand",
        "price_usd": "Average Price (USD)"
    },
    title="💰 Performance Within Reach: Brands Averaging Under $200K",
    template="simple_white"
)

# Step 3: Visual Enhancements
fig_bar.update_traces(
    marker_color="#336699",  # deep neutral blue
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig_bar.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(family="Segoe UI", size=13),
    margin=dict(l=50, r=30, t=60, b=80),
    yaxis=dict(
        title="Average Price",
        tickprefix="$",
        separatethousands=True,
        gridcolor="rgba(200,200,200,0.2)"
    ),
    xaxis=dict(
        title="",
        tickangle=-40,
        tickfont=dict(size=11),
        showgrid=False
    ),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)

fig_bar.show()


In [ ]:
from dash import Dash, dcc, html, Input, Output, State
import pandas as pd
import plotly.express as px
import datetime

#  Data Load & Prep 
df = pd.read_csv("sport_car_price_cleaned.csv")
df = df.dropna(subset=["price_usd", "horsepower", "car_make", "zero_to_60mph_s"])
df = df[df["price_usd"] > 0]
brands = sorted(df["car_make"].unique())

# Price slider marks (50k to 3M)
price_marks = {
    i: f"${i//1000}k" if i < 1_000_000 else f"${i/1_000_000:.1f}M".rstrip("0").rstrip(".")
    for i in range(50_000, 3_000_001, 250_000)
}
price_marks[3_000_000] = "$3M"

#  Dash App Init 
app = Dash(__name__)
server = app.server

#  Layout 
app.layout = html.Div([
    html.H2("Ultimate Sports Car Dashboard", style={"textAlign": "center", "marginBottom": "20px"}),

    html.Div([
        html.Label("Min Price"), 
        dcc.Slider(id="min-price-slider", min=50000, max=3000000, step=10000, value=50000,
                   marks=price_marks, tooltip={"placement": "bottom"}),

        html.Br(),
        html.Label("Max Price"),
        dcc.Slider(id="max-price-slider", min=50000, max=3000000, step=10000, value=3000000,
                   marks=price_marks, tooltip={"placement": "bottom"})
    ], style={"padding": "10px 20px"}),

    html.Div([
        html.Label("Select Brands"),
        dcc.Dropdown(id='make-dropdown', options=[{"label": b, "value": b} for b in brands],
                     multi=True, placeholder="Filter by brand")
    ], style={"padding": "0 20px", "marginBottom": "20px"}),

    html.Div(id="car-count", style={"fontWeight": "bold", "padding": "0 20px"}),
    html.Div(id="summary-stats", style={"padding": "10px 20px", "fontSize": "15px"}),

    dcc.Graph(id="hp-price-graph", style={"padding": "0 20px"}),

    html.Div([
        html.Button("⬇️ Download CSV", id="download-btn", n_clicks=0,
                    style={"backgroundColor": "#333", "color": "white", "padding": "10px 15px", "margin": "10px"}),
        html.Button("🔄 Reset", id="reset-btn", n_clicks=0,
                    style={"padding": "10px 15px", "margin": "10px"}),
        dcc.Download(id="download-dataframe-csv")
    ]),

    html.Footer("Crafted by Hari Vinayak Darga ",
                style={"textAlign": "center", "marginTop": "40px", "fontSize": "13px", "color": "#666"})
], style={"fontFamily": "Segoe UI", "backgroundColor": "#fff", "padding": "20px"})

#  Graph Updated Callback 
@app.callback(
    Output("hp-price-graph", "figure"),
    Output("car-count", "children"),
    Output("summary-stats", "children"),
    Input("min-price-slider", "value"),
    Input("max-price-slider", "value"),
    Input("make-dropdown", "value")
)
def update_graph(min_price, max_price, selected_makes):
    dff = df[(df['price_usd'] >= min_price) & (df['price_usd'] <= max_price)]
    if selected_makes:
        dff = dff[dff['car_make'].isin(selected_makes)]

    fig = px.scatter(
        dff, x="horsepower", y="price_usd", color="car_make",
        hover_data=["car_model", "year", "engine_size_l", "zero_to_60mph_s"],
        labels={"horsepower": "Horsepower", "price_usd": "Price (USD)", "car_make": "Car Make"},
        title=f"Horsepower vs. Price for {len(dff):,} Cars"
    )
    fig.update_traces(marker=dict(size=12, opacity=0.9))
    fig.update_layout(
        plot_bgcolor="#f9f9f9", paper_bgcolor="#f9f9f9",
        xaxis=dict(showgrid=True, gridcolor="#ccc"),
        yaxis=dict(showgrid=True, gridcolor="#ccc"),
        font=dict(family="Segoe UI", size=14)
    )

    avg_price = f"${int(dff['price_usd'].mean()):,}" if not dff.empty else "N/A"
    top_brand = dff['car_make'].value_counts().idxmax() if not dff.empty else "N/A"
    fastest = f"{dff['zero_to_60mph_s'].min():.2f}s" if not dff.empty else "N/A"
    stats = f"💰 Average Price: {avg_price} | 🏆 Top Brand: {top_brand} | ⚡ Fastest 0–60 mph: {fastest}"
    return fig, f"🚘 Showing {len(dff):,} cars.", stats

#  Download CSV Callback 
@app.callback(
    Output("download-dataframe-csv", "data"),
    Input("download-btn", "n_clicks"),
    State("min-price-slider", "value"),
    State("max-price-slider", "value"),
    State("make-dropdown", "value"),
    prevent_initial_call=True
)
def download_csv(n_clicks, min_price, max_price, selected_makes):
    dff = df[(df["price_usd"] >= min_price) & (df["price_usd"] <= max_price)]
    if selected_makes:
        dff = dff[dff["car_make"].isin(selected_makes)]
    filename = f"filtered_cars_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')}.csv"
    return dcc.send_data_frame(dff.to_csv, filename, index=False)

#  Reset Filters Callback 
@app.callback(
    Output("min-price-slider", "value"),
    Output("max-price-slider", "value"),
    Output("make-dropdown", "value"),
    Input("reset-btn", "n_clicks"),
    prevent_initial_call=True
)
def reset_filters(n):
    return 50000, 3000000, []

#  Run App 
if __name__ == "__main__":
    app.run_server(debug=False, port=8051)


# http://127.0.0.1:8051/



# While building this dashboard, I drew significant inspiration from a project on WRC cars created by another developer. Their work served as a helpful reference throughout. You can view their project here: https://github.com/KrzysztofLin/WRC_car_dashboard/blob/main/WRC_car_dashboard.ipynb